# OpenNMT-py によるニューラル機械翻訳演習
OpenNMT-py というツールを用いて英日機械翻訳の演習を行います。
Google Colaboratory は GPU を用いた実験ができるので、GPU を用いてニューラル機械翻訳の実験をしましょう。（「編集」→「ノートブックの設定」から「ハードウェアアクセラレータ」を GPU にしておいてください。以下、Python 3 で説明します。）
データは田中コーパスという自由に使える対訳コーパスを使用します。翻訳結果を実際に見て評価する、ということも演習の目的です。

# OpenNMT-py のインストール

## pip によるインストール
OpenNMT-py を使って実験を行います。かなり開発が速く、数週間で内容が変わってしまうので、ここでは2024年3月にリリースされた v3.5.1 というバージョンを使います。pip というのは Python のソフトウェアをインストールするときに使うツールの名前で、依存関係も含めて自動でセットアップしてくれます。

2026年10月時点の Google Colab の Python は 3.13 ですが、OpenNMT-py 3.5.1 が使うライブラリ（PyTorch 2.2 や pyonmttok など）は Python 3.12 までしか対応していません。そこで、uv という高速なパッケージ管理ツールを使って Python 3.12 の仮想環境を /content/onmt-env に作り、そこに OpenNMT-py をインストールします（1〜2分かかります）。最後に True と表示されれば GPU が使えます。その次のセルでは、作った環境のコマンドが !onmt_train のように使えるよう PATH を設定します。ランタイムを再起動した場合は、PATH のセルをもう一度実行してください。

なお、OpenNMT-py の開発は終了しています。後継の Eole を使った演習も用意しているので、[サポートサイト](https://github.com/mamorlis/nlpbook)を参照してください。

In [ ]:
!pip install -q uv
!uv venv -q --clear -p 3.12 /content/onmt-env
!uv pip install -q -p /content/onmt-env OpenNMT-py==3.5.1 "numpy<2"
!/content/onmt-env/bin/python -c "import sys, torch; print(sys.version, torch.__version__, torch.cuda.is_available())"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 122.6 MB/s eta 0:00:00
3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0] 2.2.2+cu121 True


In [ ]:
import os
os.environ["PATH"] = "/content/onmt-env/bin:" + os.environ["PATH"]
!which onmt_train

/content/onmt-env/bin/onmt_train


# 対訳データのインストールと前処理

## 田中コーパス
田中コーパスというデータで翻訳実験を行います。日本の大学生が翻訳したデータです。そのうち適当な長さの5万文だけを抽出して実験しやすいようにしてくれたデータを用います。
train.{ja,en}というのが学習データ、dev.{ja,en}というのが開発データ、test.{ja,en}というのがテストデータです。

In [ ]:
![ -d small_parallel_enja ] || git clone https://github.com/odashi/small_parallel_enja

Cloning into 'small_parallel_enja'...
remote: Enumerating objects: 35, done.
remote: Total 35 (delta 0), reused 0 (delta 0), pack-reused 35 (from 1)
Receiving objects: 100% (35/35), 1.37 MiB | 23.73 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [ ]:
!ls small_parallel_enja

dev.en     train.en.000       train.en.vocab.all  train.ja.004
dev.ja     train.en.001       train.ja            train.ja.vocab.4k
README.md  train.en.002       train.ja.000        train.ja.vocab.all
test.en    train.en.003       train.ja.001
test.ja    train.en.004       train.ja.002
train.en   train.en.vocab.4k  train.ja.003


## データの前処理
onmt_build_vocab というスクリプトでデータの前処理をします。語彙のサイズをコントロールしたり、単語分割をしたりします。YAML という形式で書かれた設定ファイルが必要なので、準備しておきます。以下では tanaka-enja.yaml という名前で設定ファイルを保存します。

In [ ]:
%%bash
cat > tanaka-enja.yaml << __EOF__
# 前処理したデータの出力先の設定
save_data: ./tanaka
src_vocab: ./tanaka/vocab.src
tgt_vocab: ./tanaka/vocab.tgt
overwrite: True

# 学習に必要なデータの置き場所
data:
        corpus_1:
                path_src: small_parallel_enja/train.en
                path_tgt: small_parallel_enja/train.ja
                weight: 1
        valid:
                path_src: small_parallel_enja/dev.en
                path_tgt: small_parallel_enja/dev.ja

# ここから学習時の設定
world_size: 1
gpu_ranks: [0]

save_model: ./tanaka/run/model
save_checkpoint_steps: 500
train_steps: 20000
valid_steps: 500
__EOF__

In [ ]:
!onmt_build_vocab -config tanaka-enja.yaml -n_sample 10000

Corpus corpus_1's weight should be given. We default it to 1 for you.
[2026-10-20 11:18:36,158 INFO] Counter vocab from 10000 samples.
[2026-10-20 11:18:36,158 INFO] Build vocab on 10000 transformed examples/corpus.
[2026-10-20 11:18:36,348 INFO] Counters src: 3447
[2026-10-20 11:18:36,348 INFO] Counters tgt: 4181


# NMTの訓練

## 学習
onmt_train というスクリプトで学習します。上述の設定ファイルで挙動を制御できます。いくつかオプションがあるので、変えてみると挙動も変わります。
world_size で使う GPU の個数、gpu_ranks で使う GPU を指定します（0スタートです）。指定しないと CPU になってしまいます。CPU は GPU の数十倍遅いです。
train_steps は学習するステップを指定します。20,000だとこのデータは15分ほどで学習できます。
デフォルトでは RNN (LSTM) を使って学習します。Transformer など他のものに変更したい場合は[設定ファイルの書き方](https://opennmt.net/OpenNMT-py/options/train.html#Model-%20Encoder-Decoder)を見て編集してみましょう。[Google の Transformer モデルの設定ファイルの書き方も公開](https://opennmt.net/OpenNMT-py/FAQ.html#how-do-i-train-the-transformer-model)されています。（ただし、今回はデータが少ないので Transformer のデフォルト設定で学習しても必ずしも精度は高くなりません）

In [ ]:
!onmt_train -config tanaka-enja.yaml

[2026-10-20 11:18:39,130 INFO] Missing transforms field for corpus_1 data, set to default: [].
[2026-10-20 11:18:39,130 WARNING] Corpus corpus_1's weight should be given. We default it to 1 for you.
[2026-10-20 11:18:39,130 INFO] Missing transforms field for valid data, set to default: [].
[2026-10-20 11:18:39,130 INFO] Parsed 2 corpora from -data.
[2026-10-20 11:18:39,130 INFO] Get special vocabs from Transforms: {'src': [], 'tgt': []}.
[2026-10-20 11:18:39,142 INFO] The first 10 tokens of the vocabs are:['<unk>', '<blank>', '<s>', '</s>', '.', 'the', 'i', 'to', 'you', 'is']
[2026-10-20 11:18:39,142 INFO] The decoder start token is: <s>
[2026-10-20 11:18:39,142 INFO] Building model...
[2026-10-20 11:18:41,589 INFO] Switching model to float32 for amp/apex_amp
[2026-10-20 11:18:41,589 INFO] Non quantized layer compute is fp32
[2026-10-20 11:18:41,846 INFO] NMTModel(
  (encoder): RNNEncoder(
    (embeddings): Embeddings(
      (make_embedding): Sequential(
        (emb_luts): Elementwise

## 翻訳モデル
学習されたモデルは model_step_10000.pt のような名前で保存されています。

In [ ]:
!ls tanaka/run

model_step_10000.pt  model_step_16000.pt  model_step_4500.pt
model_step_1000.pt   model_step_16500.pt  model_step_5000.pt
model_step_10500.pt  model_step_17000.pt  model_step_500.pt
model_step_11000.pt  model_step_17500.pt  model_step_5500.pt
model_step_11500.pt  model_step_18000.pt  model_step_6000.pt
model_step_12000.pt  model_step_18500.pt  model_step_6500.pt
model_step_12500.pt  model_step_19000.pt  model_step_7000.pt
model_step_13000.pt  model_step_19500.pt  model_step_7500.pt
model_step_13500.pt  model_step_20000.pt  model_step_8000.pt
model_step_14000.pt  model_step_2000.pt   model_step_8500.pt
model_step_14500.pt  model_step_2500.pt   model_step_9000.pt
model_step_15000.pt  model_step_3000.pt   model_step_9500.pt
model_step_1500.pt   model_step_3500.pt
model_step_15500.pt  model_step_4000.pt


# NMTによる翻訳

## デコード
`onmt_translate` というスクリプトでデコードします。`--model` で一つ前に学習した翻訳モデルのうちの1つを指定します。
`--src` で翻訳したい入力ファイル、`--output pred.txt` で結果を保存するファイルを指定します。
`--verbose` は結果を冗長に画面にも表示させるオプションです。

In [ ]:
!onmt_translate --model tanaka/run/model_step_20000.pt --src small_parallel_enja/test.en --output pred.txt --verbose

[2026-10-20 11:33:37,956 INFO] Loading checkpoint from tanaka/run/model_step_20000.pt
[2026-10-20 11:33:38,435 INFO] Loading data into the model
[2026-10-20 11:33:41,429 INFO] 
SENT 1: ['they', 'finally', 'acknowledged', 'it', 'as', 'true', '.']
PRED 1: 彼 ら は つい に それ を <unk> し た 。
PRED SCORE: -0.2539

[2026-10-20 11:33:41,429 INFO] 
SENT 2: ['he', 'didn', "'t", 'care', 'for', 'swimming', '.']
PRED 2: 彼 は 泳 ぐ の に 好 か な かっ た 。
PRED SCORE: -0.0709

[2026-10-20 11:33:41,429 INFO] 
SENT 3: ['he', 'is', 'no', 'less', 'kind', 'than', 'his', 'sister', '.']
PRED 3: 彼 は お 姉 さん と 同じ くらい 親切 だ 。
PRED SCORE: -0.1299

[2026-10-20 11:33:41,429 INFO] 
SENT 4: ['you', 'must', 'be', 'back', 'before', 'ten', '.']
PRED 4: １０ 時 前 に 帰 ら ね ば な ら な い 。
PRED SCORE: -0.1365

[2026-10-20 11:33:41,429 INFO] 
SENT 5: ['break', 'a', '<unk>', '.']
PRED 5: <unk> を 割 り なさ い 。
PRED SCORE: -0.4100

[2026-10-20 11:33:41,429 INFO] 
SENT 6: ['she', 'lives', 'next', 'door', 'to', 'us', '.']
PRED 6: 彼女 は 私 たち の 隣 に 住 ん で い る 

## BLEU による評価
正解（参照訳）のファイルを第一引数に、システムの出力をリダイレクト（<）で渡します。
sacrebleu というツールで評価するので、pip でインストールします。

In [ ]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 16.3 MB/s eta 0:00:00


In [ ]:
!sacrebleu small_parallel_enja/test.ja < pred.txt

{
 "name": "BLEU",
 "score": 37.2,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "64.3/43.7/30.6/22.4 (BP = 0.998 ratio = 0.998 hyp_len = 5626 ref_len = 5635)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}


## モデル選択

どのモデルがいいかというのは未知のデータで評価する必要があります。ここでは開発データとテストデータが用意されているので、開発データでもっとも性能がよいモデルでテストデータの翻訳をしてみます。2026年10月に試したときは、開発データでの BLEU は17000ステップのモデルが37.8で一番高く、次いで10000ステップのモデルが37.7でした。下の例では17000ステップのモデルを使っていますが、学習のたびに結果は少し変わるので、自分で学習したモデルの開発データでの結果を見て、一番よいステップに変えて試してみてください。

In [ ]:
!for step in {1..20}000; do echo ${step}; onmt_translate --model tanaka/run/model_step_${step}.pt --src small_parallel_enja/dev.en --output pred_step_${step}.txt; sacrebleu small_parallel_enja/dev.ja < pred_step_${step}.txt; done

1000
[2026-10-20 11:33:49,495 INFO] Loading checkpoint from tanaka/run/model_step_1000.pt
[2026-10-20 11:33:49,961 INFO] Loading data into the model
[2026-10-20 11:33:52,573 INFO] PRED SCORE: -0.9177, PRED PPL: 2.50 NB SENTENCES: 500
Time w/o python interpreter load/terminate:  3.0837628841400146
{
 "name": "BLEU",
 "score": 6.6,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "36.4/14.0/4.1/1.4 (BP = 0.913 ratio = 0.917 hyp_len = 5195 ref_len = 5668)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}
2000
[2026-10-20 11:33:55,314 INFO] Loading checkpoint from tanaka/run/model_step_2000.pt
[2026-10-20 11:33:55,789 INFO] Loading data into the model
[2026-10-20 11:33:58,864 INFO] PRED SCORE: -0.5921, PRED PPL: 1.81 NB SENTENCES: 500
Time w/o python interpreter load/terminate:  3.5561106204986572
{
 "name": "BLEU",
 "score": 20.6,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|v

In [ ]:
!onmt_translate --model tanaka/run/model_step_17000.pt --src small_parallel_enja/test.en --output pred.txt; sacrebleu small_parallel_enja/test.ja < pred.txt

## 結果の保存

画面に表示して保存する場合は `cat` というコマンドを使って表示させます。複数の結果を表示させたい場合は `paste` というコマンドを使います。
結果をファイルに保存する場合は `files.download` というメソッドを呼びます。
出力されていない結果は24時間で消えるので、ご注意ください。

In [ ]:
!paste pred.txt small_parallel_enja/test.{ja,en}

In [ ]:
from google.colab import files
files.download("pred.txt")